In [ ]:
import os, sys
from pathlib import Path
import pandas as pd

HERE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
LEVEL1 = HERE.parent

if str(LEVEL1) not in sys.path:
    sys.path.insert(0, str(LEVEL1))
    
from argparse import ArgumentParser
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import torch
import math
import numpy as np

from data.synthetic import sim_data
from estimation.ecm import ECM_update
from models.dps import DPS
from training.early_stopping import EarlyStopping
from training.trainer import *
from training.loss import spline_penalty_loss
from data.dataset import Dataset

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc

def create_loaders(X_train, y_train, X_val, y_val, batch_size=None):
    bs = batch_size if batch_size else len(X_train)
    
    train_ds = TensorDataset(X_train, y_train)
    val_ds = TensorDataset(X_val, y_val)
    
    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=len(X_val), shuffle=False) # Val 通常不需要 shuffle
    return train_loader, val_loader

##

### Benchmark

In [ ]:
case = 'year'
task = 'classification' if case == 'churn' else 'regression'

data_loader = Dataset(case)
data = data_loader.get_data()
X_train, X_val, X_test = data['X_train'], data['X_val'], data['X_test']
y_train, y_val, y_test = data['y_train'], data['y_val'], data['y_test']
bs = 2048

print(X_train.size())
train_loader, val_loader = create_loaders(X_train, y_train, X_val, y_val, batch_size = bs)
test_loader, _ = create_loaders(X_test, y_test, X_test, y_test, batch_size = bs)

Loading YearPredictionMSD... this might take a minute.
Data Loaded.
Train size: 463715, Test size: 51630, Features: 90
torch.Size([417343, 90])


In [ ]:
X_train_np, y_train_np = to_numpy(X_train), to_numpy(y_train)
X_val_np, y_val_np = to_numpy(X_val), to_numpy(y_val)
X_test_np, y_test_np = to_numpy(X_test), to_numpy(y_test)

print(f"Data Shapes - Train: {X_train_np.shape}, Val: {X_val_np.shape}, Test: {X_test_np.shape}")

# ==========================================
# 1. XGBoost (The Black-box Gold Standard)
# ==========================================
print("\n--- Training XGBoost ---")

xgb_model = xgb.XGBRegressor(
    n_estimators=2000,      
    learning_rate=0.05,     
    max_depth=6,            
    n_jobs=-1,              
    objective='reg:squarederror',
    tree_method='hist',     
    device="cuda" if torch.cuda.is_available() else "cpu", 
    early_stopping_rounds = 50
)

xgb_model.fit(
    X_train_np, y_train_np,
    eval_set=[(X_val_np, y_val_np)],
    verbose=100
)

y_pred_xgb = xgb_model.predict(X_test_np)
mse_xgb = get_mse(y_test_np, y_pred_xgb)
print(f"XGBoost Test MSE: {mse_xgb:.4f}")

Data Shapes - Train: (417343, 90), Val: (46372, 90), Test: (51630, 90)

--- Training XGBoost ---
[0]	validation_0-rmse:10.82161
[100]	validation_0-rmse:9.14407
[200]	validation_0-rmse:8.97960
[300]	validation_0-rmse:8.90589
[400]	validation_0-rmse:8.86443
[500]	validation_0-rmse:8.83493
[600]	validation_0-rmse:8.80931
[700]	validation_0-rmse:8.78999
[800]	validation_0-rmse:8.77192
[900]	validation_0-rmse:8.75718
[1000]	validation_0-rmse:8.74246
[1100]	validation_0-rmse:8.72957
[1200]	validation_0-rmse:8.71760
[1300]	validation_0-rmse:8.70707
[1400]	validation_0-rmse:8.69706
[1500]	validation_0-rmse:8.68930
[1600]	validation_0-rmse:8.68121
[1700]	validation_0-rmse:8.67405
[1800]	validation_0-rmse:8.66774
[1900]	validation_0-rmse:8.66125
[1999]	validation_0-rmse:8.65512
XGBoost Test MSE: 79.4335


In [ ]:
from sklearn.ensemble import RandomForestRegressor

X_train_np, y_train_np = to_numpy(X_train), to_numpy(y_train)
X_val_np, y_val_np = to_numpy(X_val), to_numpy(y_val)
X_test_np, y_test_np = to_numpy(X_test), to_numpy(y_test)


regr = RandomForestRegressor(n_estimators=1000, random_state=0)
regr.fit(X_train_np, y_train_np.ravel())

print(get_mse(regr.predict(X_test_np), y_test_np.ravel()))